# Train e15

Trains a subject 0 model based on the CNN and sentence-transformer setup of [d'Ascoli et al. (2025)](https://doi.org/10.1038/s41467-025-65499-0). Training combines D-SigLIP against T5-large word embeddings with a cross-entropy head for the 50 words plus "other". e15 uses seed 0 and 12 epochs of a 20-epoch cosine schedule. The checkpoint is selected using subject 0 validation scores. Downloads d2 from Hugging Face.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(".")
CACHE_DIR = None
CHECKPOINT = None
CACHE_VERSION = "d2"
RUN = "e15"
DURATION = 2.5
CLAMP = 5.0
TAILS = True
SOLO_P = 0.15
ISO_WEIGHT = 0.15
CE_WEIGHT = 1.0
LR_HEAD = 0.0003
WORDS_PER_BATCH = 128
LR = 3e-05
LR_CNN = 0.0001
WARMUP_STEPS = 2000
CLIP_NORM = 1.0
WEIGHT_DECAY = 0.0
ATTN_DROPOUT, FF_DROPOUT = (0.1, 0.0)
EMA_DECAY = 0.999
SCHEDULE_EPOCHS = 20
EPOCHS = 12
TIME_BUDGET_H = 8.5
VAL_EVERY = 1000
SMOOTH = 3
ABORT = (12000, 0.35)
MAX_WORDS = 32
SELECT_CORPORA = ["sherlock", "podcasts", "timit"]
NUM_WORKERS = 2
SEED = 0

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pnpl==0.2.0", "x-transformers==2.31.10", "huggingface-hub"], check=True
)

In [ ]:
from huggingface_hub import snapshot_download

if CACHE_DIR is None:
    CACHE_DIR = Path(snapshot_download(
        repo_id="JatinArutla/PNPL-2026-D2-Cache",
        repo_type="dataset",
        allow_patterns=['meta_d2.npz', 'meg_d2.bin', 'hold_d2.bin', 't5_d2.bin'],
    ))

In [ ]:
import copy
import json
import math
import time
from collections import deque
from importlib.metadata import version
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from x_transformers import Encoder
from pnpl.competition import load_vocabulary, write_submission

def find_input(filename):
    matches = [p for p in Path("/kaggle/input").rglob(filename) if p.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {filename} in the attached inputs, found {len(matches)}. "
            "Attach the required dataset or set its path in the first cell."
        )
    return matches[0]


CACHE_DIR = find_input("meta_d2.npz").parent if CACHE_DIR is None else Path(CACHE_DIR)
if CHECKPOINT is not None:
    CHECKPOINT = Path(CHECKPOINT)
    if CHECKPOINT.parent == Path("."):
        CHECKPOINT = find_input(CHECKPOINT.name)

for filename in (
    f"meta_{CACHE_VERSION}.npz",
    f"meg_{CACHE_VERSION}.bin",
    f"hold_{CACHE_VERSION}.bin",
    f"t5_{CACHE_VERSION}.bin",
):
    path = CACHE_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing d2 input: {path}")
if CHECKPOINT is not None and not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT}")

device = "cuda"
assert torch.cuda.is_available(), "Select a T4 GPU."
autocast_settings = dict(device_type="cuda", dtype=torch.float16)
T0 = time.time()
print(
    " | ".join(
        (
            f"{name} {version(name)}"
            for name in ("pnpl", "x-transformers", "torch", "numpy")
        )
    )
)
metadata = np.load(CACHE_DIR / f"meta_{CACHE_VERSION}.npz", allow_pickle=True)
FREQ = float(metadata["freq"])
window_samples = int(round(DURATION * FREQ))
train_meg = np.memmap(
    CACHE_DIR / f"meg_{CACHE_VERSION}.bin", dtype=np.float16, mode="r"
).reshape(-1, 306)
holdout_meg = np.memmap(
    CACHE_DIR / f"hold_{CACHE_VERSION}.bin", dtype=np.float16, mode="r"
).reshape(-1, 306)
word_embeddings = np.memmap(
    CACHE_DIR / f"t5_{CACHE_VERSION}.bin", dtype=np.float16, mode="r"
).reshape(-1, int(metadata["t5_dim"]))
assert len(train_meg) == int(metadata["n_samples"]), (
    len(train_meg),
    int(metadata["n_samples"]),
)
vocabulary = load_vocabulary("primary")
sensor_positions = torch.tensor(metadata["positions"], dtype=torch.float32).to(device)
CONFIG = dict(
    run=RUN,
    cache=CACHE_VERSION,
    duration=DURATION,
    clamp=CLAMP,
    tails=TAILS,
    solo_p=SOLO_P,
    iso_weight=ISO_WEIGHT,
    ce_weight=CE_WEIGHT,
    lr_head=LR_HEAD,
    words_per_batch=WORDS_PER_BATCH,
    lr=LR,
    lr_cnn=LR_CNN,
    warmup_steps=WARMUP_STEPS,
    clip_norm=CLIP_NORM,
    weight_decay=WEIGHT_DECAY,
    attn_dropout=ATTN_DROPOUT,
    ff_dropout=FF_DROPOUT,
    ema_decay=EMA_DECAY,
    schedule_epochs=SCHEDULE_EPOCHS,
    epochs=EPOCHS,
    seed=SEED,
)
print(CONFIG)
print(
    f"Cache: {CACHE_DIR} | {len(train_meg) / FREQ / 3600:.1f} hours | {len(metadata['onset'])} words"
)

In [ ]:
onset, run = (metadata["onset"].astype(np.int64), metadata["run"].astype(np.int64))
sentence, label = (
    metadata["sentence"].astype(np.int64),
    metadata["label"].astype(np.int64),
)
word_id, stim_id = (
    metadata["word_id"].astype(np.int64),
    metadata["stim_id"].astype(np.int64),
)
corpus, session = (metadata["run_corpus"][run], metadata["run_session"][run])
task, partition = (metadata["run_task"][run], metadata["run_partition"][run])
assert not (partition == "test").any(), "the cache contains test-partition runs"
TIMIT_TEST_SPEAKERS = set(
    "mdab0 mwbt0 felc0 mtas1 mwew0 fpas0 mjmp0 mlnt0 fpkt0 mlll0 mtls0 fjlm0 mbpm0 mklt0 fnlp0 mcmj0 mjdh0 fmgd0 mgrt0 mnjm0 fdhc0 mjln0 mpam0 fmld0".split()
)
TIMIT_VAL_SPEAKERS = set(
    "faks0 fdac1 fjem0 mgwt0 mjar0 mmdb1 mmdm2 mpdf0 fcmh0 fkms0 mbdg0 mbwm0 mcsh0 fadg0 fdms0 fedw0 mgjf0 mglb0 mrtk0 mtaa0 mtdt0 mthc0 mwjg0 fnmr0 frew0 fsem0 mbns0 mmjr0 mdls0 mdlf0 mdvc0 mers0 fmah0 fdrw0 mrcs0 mrjm4 fcal1 mmwh0 fjsj0 majc0 mjsw0 mreb0 fgjd0 fjmg0 mroa0 mteb0 mjfc0 mrjr0 fmml0 mrws1".split()
)


def timit_speaker_and_sentence(stimuli):
    speakers = []
    sentences = []
    for stimulus in stimuli:
        parts = str(stimulus).replace("\\", "/").split("/")
        stem = parts[-1].rsplit(".", 1)[0]
        if "_" in stem:
            speaker, sentence = stem.split("_", 1)
        elif len(parts) > 1:
            speaker, sentence = parts[-2], stem
        else:
            speaker, sentence = "", ""
        speakers.append(speaker.lower())
        sentences.append(sentence.upper())
    return np.array(speakers), np.array(sentences)


TRAIN, VAL, DROP = (0, 1, -1)
split = np.full(len(onset), TRAIN, dtype=np.int8)
split[(task == "Sherlock1") & (session == "11")] = VAL
split[(corpus == "podcasts") & (session == "29")] = VAL
split[(corpus == "mocha") & (session == "2")] = VAL
spk_of, sen_of = timit_speaker_and_sentence(metadata["stims"])
wspk, wsen = (spk_of[stim_id], sen_of[stim_id])
is_timit = corpus == "timit"
assert (
    wspk[is_timit] != ""
).all(), "cannot parse a speaker out of the TIMIT stimulus ids"
split[is_timit & np.isin(wspk, list(TIMIT_TEST_SPEAKERS))] = DROP
split[
    is_timit & np.isin(wspk, list(TIMIT_VAL_SPEAKERS)) & ~np.char.startswith(wsen, "SA")
] = VAL
val_sentences = set(wsen[(split == VAL) & is_timit])
split[is_timit & (split == TRAIN) & np.isin(wsen, list(val_sentences))] = DROP
for name, s in (("train", TRAIN), ("val", VAL), ("dropped", DROP)):
    print(f"{name:8s} {(split == s).sum():7d} words")
sid = run * (sentence.max() + 2) + (sentence + 1)
lo = metadata["run_offset"][run]
ok = (onset >= lo) & (onset + window_samples <= lo + metadata["run_len"][run])
print(f"{(~ok).sum()} words dropped at recording edges")


def group_sentences(mask, cap=MAX_WORDS):
    rows = np.flatnonzero(mask & ok)
    order = np.lexsort((onset[rows], sid[rows]))
    rows = rows[order]
    starts = np.flatnonzero(np.diff(sid[rows], prepend=-1))
    ends = list(starts[1:]) + [len(rows)]
    groups = []
    for start, end in zip(starts, ends):
        for offset in range(start, end, cap):
            groups.append(rows[offset : min(offset + cap, end)])
    return groups


train_g, val_g = (group_sentences(split == TRAIN), group_sentences(split == VAL))
val_short = [g for g in group_sentences(split == VAL, cap=10**9) if 3 <= len(g) <= 8]
h_epoch, h_at = (
    metadata["hold_epoch"].astype(np.int64),
    metadata["hold_at"].astype(np.int64),
)
h_source = metadata["hold_source"].astype(np.int8)
h_start, h_len = (
    metadata["hold_start"].astype(np.int64),
    metadata["hold_len"].astype(np.int64),
)
HOLD_GROUPS = [np.flatnonzero(h_epoch == e) for e in np.unique(h_epoch)]
HOLD_ROOM = h_start[h_epoch] + h_len[h_epoch] - h_at
HOLD_TAILS = np.array([HOLD_ROOM[g[-1]] for g in HOLD_GROUPS if h_source[g[0]] == 0])
ISO_LEN = int(np.median(HOLD_ROOM[h_source == 1]))
for g in HOLD_GROUPS:
    assert (
        np.diff(h_at[g]) >= 0
    ).all(), "holdout rows are not in onset order inside an epoch"
print(
    f"train {len(train_g)} groups / {sum(map(len, train_g))} words | val {len(val_g)} / {sum(map(len, val_g))} | short val sentences {len(val_short)}"
)
print(
    f"holdout: tails {np.percentile(HOLD_TAILS, [10, 50, 90]) / FREQ} s (p10 p50 p90) | isolated epochs {ISO_LEN / FREQ:g} s | rows with a full {DURATION:g} s window {(HOLD_ROOM >= window_samples).mean():.1%}"
)

In [ ]:
class FourierEmb(nn.Module):
    def __init__(self, dimension=2048, margin=0.2):
        super().__init__()
        self.dimension = dimension
        self.margin = margin

    def forward(self, positions):
        leading_shape = positions.shape[:-1]
        count = int((self.dimension // 2) ** 0.5)
        frequencies = torch.arange(
            count, device=positions.device, dtype=positions.dtype
        )
        width = 1 + 2 * self.margin
        x_frequencies = 2 * math.pi * frequencies[:, None] / width
        y_frequencies = 2 * math.pi * frequencies / width
        shifted = (positions + self.margin)[..., None, None, :]
        angles = shifted[..., 0] * x_frequencies + shifted[..., 1] * y_frequencies
        angles = angles.reshape(*leading_shape, -1)
        return torch.cat([torch.cos(angles), torch.sin(angles)], dim=-1)


class ChannelMerger(nn.Module):
    def __init__(self, chout=270, pos_dim=2048, dropout=0.2):
        super().__init__()
        self.heads = nn.Parameter(torch.randn(1, chout, pos_dim) / pos_dim**0.5)
        self.embedding = FourierEmb(pos_dim)
        self.dropout = dropout

    def forward(self, meg, positions):
        scores = self.embedding(positions) @ self.heads[0].T
        if self.training and self.dropout:
            centre = torch.rand(2, device=meg.device)
            excluded = (positions - centre).norm(dim=-1) <= self.dropout
            scores = scores.masked_fill(excluded[:, None], float("-inf"))
        weights = torch.softmax(scores.float(), 0).to(meg.dtype)
        return torch.einsum("bct,co->bot", meg, weights)


class ConvBlock(nn.Module):
    def __init__(self, chin, chout, dilation, last=False, glu=False, drop_in=0.0):
        super().__init__()
        layers = []
        if drop_in:
            layers.append(nn.Dropout(drop_in))
        layers.append(nn.Conv1d(chin, chout, 3, 1, dilation, dilation=dilation))
        if not last:
            layers.extend([nn.BatchNorm1d(chout), nn.GELU()])
        self.block = nn.Sequential(*layers)
        self.glu = None
        if glu:
            self.glu = nn.Sequential(
                nn.Conv1d(chout, 2 * chout, 3, padding=1), nn.GLU(dim=1)
            )

    def forward(self, x):
        output = self.block(x)
        if output.shape == x.shape:
            output = output + x
        if self.glu is not None:
            output = self.glu(output)
        return output


class BrainModule(nn.Module):
    def __init__(self, out_dim=1024, hidden=160, depth=5, initial=512, agg_hidden=256):
        super().__init__()
        self.merger = ChannelMerger()
        self.initial = nn.Conv1d(270, initial, 1)
        self.subject = nn.Conv1d(initial, initial, 1, bias=False)
        sizes = [initial] + [hidden] * (depth - 1) + [out_dim]
        blocks = []
        for layer in range(depth):
            blocks.append(
                ConvBlock(
                    sizes[layer],
                    sizes[layer + 1],
                    2**layer,
                    last=layer == depth - 1,
                    glu=(layer + 1) % 2 == 0,
                    drop_in=0.1 if layer == 0 else 0.0,
                )
            )
        self.encoder = nn.Sequential(*blocks)
        self.att = nn.Sequential(
            nn.Conv1d(out_dim, agg_hidden, 1), nn.Tanh(), nn.Conv1d(agg_hidden, 1, 1)
        )

    def forward(self, x, positions):
        x = self.merger(x, positions)
        x = self.initial(x)
        x = self.subject(x)
        x = self.encoder(x)
        attention = torch.softmax(self.att(x), dim=-1)
        return (x * attention).sum(-1)


class Decoder(nn.Module):

    def __init__(self, dim=1024):
        super().__init__()
        self.cnn = BrainModule(out_dim=dim)
        self.cls = nn.Linear(dim, 51)
        self.transformer = Encoder(
            dim=dim,
            depth=16,
            heads=16,
            rotary_pos_emb=True,
            attn_dropout=ATTN_DROPOUT,
            ff_dropout=FF_DROPOUT,
            use_scalenorm=True,
            scale_residual=True,
        )
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
        self.bias = nn.Parameter(torch.tensor(-10.0))

    def forward(self, real, positions, mask):
        emb = F.normalize(self.cnn(real, positions), dim=-1)
        h = emb.new_zeros(*mask.shape, emb.shape[-1])
        h = h.masked_scatter(mask[..., None], emb)
        return F.normalize(self.transformer(h, mask=mask), dim=-1)[mask]

In [ ]:
def dsiglip(predictions, targets, word_ids, scale, bias):
    logits = predictions @ targets.T * scale + bias
    positive = torch.eye(len(predictions), dtype=torch.bool, device=predictions.device)
    keep = (word_ids[:, None] != word_ids[None, :]) | positive
    signs = torch.where(positive, 1.0, -1.0)
    return -(F.logsigmoid(signs * logits) * keep).sum() / len(predictions)


def macro_recall(scores, labels, k=10):
    top_words = scores.topk(k, dim=1).indices.numpy()
    hit = (top_words == np.asarray(labels)[:, None]).any(1)
    class_recalls = []
    for word in range(50):
        selected = labels == word
        if selected.any():
            class_recalls.append(hit[selected].mean())
    return float(np.mean(class_recalls))


def normalise_word(w):
    w = str(w).replace("’", "'").lower()
    return "".join((c for c in w if c.isalnum() or c in "-'"))


word_row = {str(w): i for i, w in enumerate(metadata["words"])}
cand_rows = [word_row[normalise_word(v)] for v in vocabulary]
assert [str(metadata["words"][i]) for i in cand_rows] == [
    normalise_word(v) for v in vocabulary
]
candidate_embeddings = F.normalize(
    torch.tensor(np.asarray(word_embeddings[cand_rows], dtype=np.float32)), dim=-1
).to(device)

In [ ]:
def raw_windows(store, starts, rooms):
    windows = np.zeros((len(starts), window_samples, 306), dtype=np.float16)
    for row, (start, available) in enumerate(zip(starts, rooms)):
        length = min(window_samples, int(available))
        windows[row, :length] = store[start : start + length]
    return windows


def scale_windows(raw, rooms, chunk=256):
    batches = []
    for windows, lengths in zip(raw.float().split(chunk), rooms.split(chunk)):
        valid = (
            torch.arange(window_samples, device=windows.device)[None] < lengths[:, None]
        )
        valid = valid[..., None]
        quantiles = torch.nanquantile(
            windows.masked_fill(~valid, float("nan")),
            torch.tensor([0.25, 0.5, 0.75], device=windows.device),
            dim=1,
        )
        median = quantiles[1][:, None]
        spread = (quantiles[2] - quantiles[0]).clamp_min(1e-6)[:, None]
        windows = (windows - median) / spread
        windows = windows.clamp(-CLAMP, CLAMP) * valid
        batches.append(windows.transpose(1, 2))
    return torch.cat(batches)


def tail_rooms(g, tail):
    return np.minimum(window_samples, onset[g[-1]] - onset[g] + tail)


class Groups(torch.utils.data.Dataset):
    def __init__(self, store, at, groups, rooms, with_targets=True):
        self.store = store
        self.at = at
        self.groups = groups
        self.rooms = rooms
        self.with_targets = with_targets
        self.rng = None

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, index):
        if self.rng is None:
            worker = torch.utils.data.get_worker_info()
            worker_id = worker.id if worker else 0
            self.rng = np.random.default_rng([SEED, worker_id, int(time.time())])
        rows = self.groups[index]
        if callable(self.rooms):
            lengths = self.rooms(rows, self.rng)
        else:
            lengths = self.rooms[index]
        windows = raw_windows(self.store, self.at[rows], lengths)
        item = [
            torch.from_numpy(windows),
            torch.from_numpy(np.asarray(lengths, dtype=np.int64)),
        ]
        if self.with_targets:
            item.extend(
                [
                    torch.from_numpy(
                        np.asarray(word_embeddings[word_id[rows]], dtype=np.float32)
                    ),
                    torch.from_numpy(word_id[rows]),
                    torch.from_numpy(label[rows]),
                ]
            )
        return item


def collate(items):
    batch_size = len(items)
    max_words = max(len(item[0]) for item in items)
    raw = torch.zeros(batch_size, max_words, window_samples, 306, dtype=torch.float16)
    rooms = torch.ones(batch_size, max_words, dtype=torch.long)
    mask = torch.zeros(batch_size, max_words, dtype=torch.bool)
    for row, item in enumerate(items):
        count = len(item[0])
        raw[row, :count] = item[0]
        rooms[row, :count] = item[1]
        mask[row, :count] = True
    batch = [raw, rooms, mask]
    for field in range(2, len(items[0])):
        batch.append(torch.cat([item[field] for item in items]))
    return batch


def loader(ds, batch, shuffle=False):
    return DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        drop_last=shuffle,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=device == "cuda",
    )


PER_BATCH = max(1, int(round(WORDS_PER_BATCH / np.mean([len(g) for g in train_g]))))


def train_rooms(rows, rng):
    return tail_rooms(rows, rng.choice(HOLD_TAILS))


train_dl = loader(
    Groups(train_meg, onset, train_g, train_rooms), PER_BATCH, shuffle=True
)
train_words = np.concatenate(train_g)
solo_dl = loader(
    Groups(
        train_meg,
        onset,
        [np.array([i]) for i in train_words],
        lambda g, rng: np.array([ISO_LEN]),
    ),
    WORDS_PER_BATCH,
    shuffle=True,
)
steps_per_epoch = int(len(train_dl) * (1 + SOLO_P))
solo_iter = None


def training_batches(rng):
    global solo_iter
    for batch in train_dl:
        yield batch
        if rng.random() < SOLO_P:
            if solo_iter is None:
                solo_iter = iter(solo_dl)
            try:
                yield next(solo_iter)
            except StopIteration:
                solo_iter = iter(solo_dl)
                yield next(solo_iter)


_rng = np.random.default_rng(0)


def validation_rooms(groups):
    return [tail_rooms(g, _rng.choice(HOLD_TAILS)) for g in groups]


val_iso = [np.array([i]) for i in np.concatenate(val_g) if label[i] >= 0]
SURFACES = {
    "served": (val_g, validation_rooms(val_g)),
    "iso": (val_iso, [np.array([ISO_LEN])] * len(val_iso)),
    "full": (val_g, [np.full(len(g), window_samples) for g in val_g]),
    "short": (val_short, validation_rooms(val_short)),
}
print(
    f"{PER_BATCH} sentences per batch | about {steps_per_epoch} steps per epoch, {SOLO_P:.0%} of them single-word batches | {EPOCHS} of a {SCHEDULE_EPOCHS}-epoch cosine"
)
for k, (g, _) in SURFACES.items():
    print(
        f"  {k:7s} {len(g):5d} groups | {int((label[np.concatenate(g)] >= 0).sum()):5d} scored rows"
    )

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
model = Decoder().to(device)


def load_weights(net, path):
    checkpoint = torch.load(path, map_location=device)
    for name, value in CONFIG.items():
        assert (
            checkpoint["config"][name] == value
        ), f"Checkpoint setting differs: {name}"
    net.load_state_dict(checkpoint["state_dict"])


with torch.no_grad():
    model.cls.weight.zero_()
    model.cls.weight[:50] = 10.0 * candidate_embeddings
    model.cls.bias.zero_()
n = np.bincount(np.where(label[train_words] < 0, 50, label[train_words]), minlength=51)
CLASS_W = torch.tensor((n.mean() / n) ** 0.5, dtype=torch.float32, device=device)
print(
    f"cross-entropy head on | class weights {CLASS_W[:50].min():.2f} to {CLASS_W[:50].max():.2f}, other {CLASS_W[50]:.2f}"
)
cnn_params = list(model.cnn.parameters())
head = list(model.cls.parameters())
other = [p for n, p in model.named_parameters() if not n.startswith(("cnn.", "cls."))]
opt = torch.optim.AdamW(
    [
        dict(params=cnn_params, lr=LR_CNN),
        dict(params=other, lr=LR),
        dict(params=head, lr=LR_HEAD),
    ],
    weight_decay=WEIGHT_DECAY,
)
total = SCHEDULE_EPOCHS * steps_per_epoch


def lr_factor(s):
    if s < WARMUP_STEPS:
        return (s + 1) / WARMUP_STEPS
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(s, total) / total))


sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
scaler = torch.amp.GradScaler(device)
ema = copy.deepcopy(model).eval()


@torch.no_grad()
def ema_update(step):
    decay = min(EMA_DECAY, (1 + step) / (10 + step))
    for average, current in zip(ema.state_dict().values(), model.state_dict().values()):
        if average.dtype.is_floating_point:
            average.lerp_(current, 1 - decay)
        else:
            average.copy_(current)


def to_device(batch):
    return [t.to(device, non_blocking=True) for t in batch]

In [ ]:
def zrow(s):
    return (s - s.mean(1, keepdim=True)) / s.std(1, keepdim=True).clamp_min(1e-06)


READOUT = "both"


@torch.inference_mode()
def score_rows(net, store, at, groups, rooms):
    net.eval()
    batch_size = max(
        1, int(256 / np.mean([len(group_sentences) for group_sentences in groups]))
    )
    retrieval_scores = []
    classifier_scores = []
    dataset = Groups(store, at, groups, rooms, with_targets=False)
    for raw, lengths, mask in loader(dataset, batch_size):
        raw, lengths, mask = to_device([raw, lengths, mask])
        windows = scale_windows(raw[mask], lengths[mask])
        with torch.autocast(**autocast_settings):
            embeddings = net(windows, sensor_positions, mask).float()
        retrieval_scores.append((embeddings @ candidate_embeddings.T).cpu())
        classifier_scores.append(net.cls(embeddings)[:, :50].float().cpu())
    scores = {"ret": torch.cat(retrieval_scores), "cls": torch.cat(classifier_scores)}
    scores["both"] = zrow(scores["ret"]) + zrow(scores["cls"])
    return scores


def validate(net, surface, stash=None):
    groups, rooms = SURFACES[surface]
    rows = np.concatenate(groups)
    keep = label[rows] >= 0
    labels = label[rows][keep]
    corpora = corpus[rows][keep]
    results = {}
    if stash is not None:
        stash[f"{surface}_y"] = labels
        stash[f"{surface}_corpus"] = corpora
    for readout, scores in score_rows(net, train_meg, onset, groups, rooms).items():
        scores = scores[keep]
        if stash is not None:
            stash[f"{surface}_{readout}"] = scores.numpy()
        metrics = {"all": macro_recall(scores, labels)}
        for name in np.unique(corpora):
            selected = corpora == name
            metrics[name] = macro_recall(scores[selected], labels[selected])
        selected_scores = [metrics[name] for name in SELECT_CORPORA if name in metrics]
        metrics["select"] = float(np.mean(selected_scores))
        results[readout] = metrics
    return results

In [ ]:
net = ema
if CHECKPOINT is not None:
    ckpt = CHECKPOINT
    load_weights(net, ckpt)
    best, step = None, 0
    print(f"Loaded {ckpt}; training steps in this run: 0")
else:
    recent, best, step, stop = (deque(maxlen=SMOOTH), -1.0, 0, False)
    epoch_rng = np.random.default_rng(SEED + 1)
    for epoch in range(1, EPOCHS + 1):
        if stop:
            break
        for batch in training_batches(epoch_rng):
            model.train()
            raw, rooms, mask, targ, ids, lab = to_device(batch)
            real = scale_windows(raw[mask], rooms[mask])
            with torch.autocast(**autocast_settings):
                h = model(real, sensor_positions, mask).float()
                loss = dsiglip(
                    h,
                    F.normalize(targ, dim=-1),
                    ids,
                    model.logit_scale.exp(),
                    model.bias,
                )
                loss = loss + CE_WEIGHT * F.cross_entropy(
                    model.cls(h).float(), torch.where(lab < 0, 50, lab), weight=CLASS_W
                )
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            ema_update(step)
            if step % VAL_EVERY == 0:
                every, every_iso = (validate(net, "served"), validate(net, "iso"))
                served, iso = (every[READOUT], every_iso[READOUT])
                heads = "".join(
                    (f" | {k} {v['all']:.4f}" for k, v in every.items() if k != READOUT)
                )
                recent.append(
                    (1 - ISO_WEIGHT) * served["select"] + ISO_WEIGHT * iso["all"]
                )
                score, star = (sum(recent) / len(recent), "")
                if len(recent) == SMOOTH and score > best:
                    best, star = (score, " *")
                    torch.save(
                        {"state_dict": net.state_dict(), "config": CONFIG},
                        OUTPUT_DIR / f"best_{RUN}.pt",
                    )
                print(
                    f"  ep {epoch} step {step:6d} | loss {float(loss.detach()):.4f} | lr {sched.get_last_lr()[1]:.2e} | served {served['all']:.4f} "
                    + " ".join(
                        (
                            f"{k}={served[k]:.3f}"
                            for k in sorted(served)
                            if k not in ("all", "select")
                        )
                    )
                    + heads
                    + f" | iso {iso['all']:.4f} | select {recent[-1]:.4f} | {(time.time() - T0) / 3600:.2f} h{star}",
                    flush=True,
                )
                if step >= ABORT[0] and served["all"] < ABORT[1]:
                    print(
                        f"served validation is {served['all']:.3f} at step {step}: the run has not taken off. Stopping to save quota."
                    )
                    stop = True
                if (time.time() - T0) / 3600 > TIME_BUDGET_H:
                    print("time budget reached")
                    stop = True
                if stop:
                    break
    assert step > 0, "no training steps ran"
    ckpt = OUTPUT_DIR / f"best_{RUN}.pt"
    if not ckpt.exists():
        torch.save({"state_dict": net.state_dict(), "config": CONFIG}, ckpt)
        print(
            f"no checkpoint was selected in {step} steps; saved the final weights instead"
        )
    load_weights(net, ckpt)

In [ ]:
if CHECKPOINT is None:
    print(
        f"selected on smoothed (1 - {ISO_WEIGHT}) x served[{'+'.join(SELECT_CORPORA)}] + {ISO_WEIGHT} x iso: {best:.4f}\n"
    )
else:
    print(f"Validation and holdout predictions from {ckpt}")
stash = {}
final = {k: validate(net, k, stash) for k in SURFACES}
np.savez_compressed(OUTPUT_DIR / f"val_scores_{RUN}.npz", **stash)
for k, by_readout in final.items():
    for name, parts in by_readout.items():
        print(
            f"{k:7s} {name:5s} "
            + " ".join((f"{n}={v:.4f}" for n, v in sorted(parts.items())))
        )
F_ISO = float((h_source == 1).mean())
predicted = (1 - F_ISO) * final["served"][READOUT]["all"] + F_ISO * final["iso"][
    READOUT
]["all"]
print(f"Validation estimate: {predicted:.4f}")
order = np.argsort(np.concatenate(HOLD_GROUPS))
by_readout = {
    k: v[order]
    for k, v in score_rows(
        net, holdout_meg, h_at, HOLD_GROUPS, [HOLD_ROOM[g] for g in HOLD_GROUPS]
    ).items()
}
scores = by_readout[READOUT]
np.save(OUTPUT_DIR / f"holdout_scores_{RUN}.npy", scores.numpy())
for k, v in by_readout.items():
    if k != READOUT:
        np.save(OUTPUT_DIR / f"holdout_scores_{RUN}_{k}.npy", v.numpy())
ranks = scores.argsort(1).argsort(1).double() + 1
SUB = OUTPUT_DIR / f"submission_deep_{RUN}.csv"
write_submission(
    str(SUB),
    indices=metadata["hold_indices"],
    primary_probs=(ranks / ranks.sum(1, keepdim=True)).numpy(),
)
top1 = scores.argmax(1).numpy()
for name, sel in (("with context", h_source == 0), ("single words", h_source == 1)):
    print(
        f"{name:14s} {int(sel.sum()):4d} rows | distinct top-1 {len(np.unique(top1[sel])):2d}/50"
    )
(OUTPUT_DIR / f"run_{RUN}.json").write_text(
    json.dumps(
        {
            **CONFIG,
            "checkpoint_input": str(CHECKPOINT) if CHECKPOINT is not None else None,
            "val_select": best,
            "steps": step,
            "predicted_board": predicted,
            "final": final,
            "hours": round((time.time() - T0) / 3600, 2),
        },
        indent=1,
    )
)
print(
    f"\nwrote {SUB.name}, holdout_scores_{RUN}.npy and run_{RUN}.json | total {(time.time() - T0) / 3600:.2f} h"
)